# Curves and overfitting: bias versus variance
**Machine Learning · Week 2 (Regression and gradient descent) · Day 4**

Lesson page: https://sara-aminpour.com/tutorials/ml/w2d4.html

A straight line is a polynomial of degree 1. Higher degrees can bend more.
Today: how bendy is too bendy, and why.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

def true_curve(x):
    return np.sin(np.pi * (x + 1))

def make_data(n):
    x = rng.uniform(-1, 1, n)
    return x, true_curve(x) + rng.normal(0, 0.3, n)     # noise with spread 0.3

x_train, y_train = make_data(15)       # small training set
x_val, y_val = make_data(300)          # lots of fresh data to measure the real error
print("training points:", len(x_train), " validation points:", len(x_val))

## Polynomial features: still a linear model
$w_0 + w_1x + w_2x^2 + w_3x^3$ is a **linear** model in the new features $(1, x, x^2, x^3)$.
So we build those columns and use ordinary least squares, exactly like the pizza line.

In [ ]:
def poly_features(x, degree):
    return np.vander(x, degree + 1, increasing=True)     # columns 1, x, x^2, ..., x^degree

def fit(x, y, degree):
    w, *_ = np.linalg.lstsq(poly_features(x, degree), y, rcond=None)
    return w

def mse(w, x, y):
    return np.mean((poly_features(x, len(w) - 1) @ w - y) ** 2)

print(poly_features(np.array([0.5, 2.0]), 3))

In [ ]:
grid = np.linspace(-1, 1, 300)
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4), sharey=True, layout="constrained")
for ax, degree in zip(axes, [1, 4, 12]):
    w = fit(x_train, y_train, degree)
    ax.plot(grid, true_curve(grid), "--", color="#bcb7cb", label="true curve")
    ax.plot(grid, poly_features(grid, degree) @ w, color="#7b5cfa", lw=2.5, label="fit")
    ax.scatter(x_train, y_train, color="#ff8a50", zorder=3, s=25)
    ax.set_ylim(-2, 2)
    ax.set_title(f"degree {degree}\ntrain {mse(w, x_train, y_train):.3f} · validation {mse(w, x_val, y_val):.2f}")
axes[0].legend(loc="lower left")
plt.show()

## Bias and variance, measured
Draw 300 different training sets of 25 points, fit each one, and look at the predictions:
**bias²** = how far the *average* fit is from the truth; **variance** = how much the fits disagree.

In [ ]:
degrees = range(0, 8)
bias2, variance = [], []
for d in degrees:
    preds = []
    for _ in range(300):
        x, y = make_data(25)
        preds.append(poly_features(grid, d) @ fit(x, y, d))
    preds = np.array(preds)                                  # 300 fits x 300 grid points
    bias2.append(np.mean((preds.mean(axis=0) - true_curve(grid)) ** 2))
    variance.append(np.mean(preds.var(axis=0)))

bias2, variance = np.array(bias2), np.array(variance)
plt.figure(figsize=(6.4, 3.6))
plt.plot(degrees, bias2, "o-", color="#ff8a50", label="bias² (too simple)")
plt.plot(degrees, variance, "o-", color="#ff8fb8", label="variance (too wiggly)")
plt.plot(degrees, bias2 + variance + 0.09, "o-", color="#7b5cfa", lw=2.5, label="expected error = bias² + variance + noise")
plt.yscale("log")
plt.xlabel("polynomial degree")
plt.ylabel("error (log scale)")
plt.legend()
plt.show()
for d in [1, 3, 7]:
    print(f"degree {d}: bias² {bias2[d]:.3f}   variance {variance[d]:.3f}   expected error {bias2[d] + variance[d] + 0.09:.3f}")

## Your turn: does more data cure overfitting?
Fit a degree-9 polynomial to training sets of 15, 50 and 500 points and compare the validation error.

In [ ]:
for n in [15, 50, 500]:
    x, y = make_data(n)
    w = np.zeros(10)          # TODO: replace with the degree-9 fit to (x, y)
    print(f"n = {n:3d}: validation MSE = {mse(w, x_val, y_val):.3f}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
for n in [15, 50, 500]:
    x, y = make_data(n)
    w = fit(x, y, 9)
    print(f"n = {n:3d}: validation MSE = {mse(w, x_val, y_val):.3f}")
print("More data pins the wiggles down: the variance shrinks, and the same flexible model stops overfitting.")
print("(The best possible is the noise level, 0.3² = 0.09.)")